# 原生交互元素

学习目标：能编写折叠内容、对话框和弹出层，区分模态、隐藏与停用，并检查关闭方式和键盘焦点。

前置知识：HTML 元素与属性、布尔属性、id、按钮与表单，以及浏览器的基本键盘操作。

适用范围：WHATWG HTML Living Standard。details 的 name、按钮的 commandfor、dialog 的 closedby、popover 的 hint 和 hidden 的 until-found 需分别检查浏览器支持，不能由“支持 dialog”推断全部可用。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/12-native-interactive-elements/。

1. [details.html](scripts/12-native-interactive-elements/details.html)：独立折叠内容与互斥组。
2. [dialog-command.html](scripts/12-native-interactive-elements/dialog-command.html)：声明式模态对话框与关闭策略。
3. [dialog-open.html](scripts/12-native-interactive-elements/dialog-open.html)：非模态 open 与表单关闭。
4. [dialog-api.html](scripts/12-native-interactive-elements/dialog-api.html)：用浏览器 API 对照两种打开方式。
5. [popover-auto.html](scripts/12-native-interactive-elements/popover-auto.html)、[popover-manual.html](scripts/12-native-interactive-elements/popover-manual.html)：弹出层的关闭方式及焦点恢复。
6. [hidden-inert.html](scripts/12-native-interactive-elements/hidden-inert.html)：隐藏、停用、恢复操作与查找后显示。

## 打开配套页面

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/html
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8012 --bind 127.0.0.1
```

Step 3：打开[折叠内容示例](http://127.0.0.1:8012/scripts/12-native-interactive-elements/details.html)。

服务根目录为 content/Web与应用开发/html；Notebook 配套链接相对于本 Notebook，页面内的相对 URL 按页面地址解析。

优先使用 HTML 声明式按钮。少量辅助脚本随页提供，无需先掌握其完整 JavaScript 语法；声明式对话框按钮不可用时，可对照第 4 节的 API 页面，但不能据此认定 closedby 已受支持。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 标签总览

| 标签 | 中文名称／含义 | 用途 |
| --- | --- | --- |
| &lt;details&gt; | 可展开的补充内容 | 按需显示说明或控件 |
| &lt;summary&gt; | 摘要与展开入口 | 概括展开后能看到什么 |
| &lt;dialog&gt; | 对话框 | 展示需要处理的提示或任务 |

popover、hidden、inert 是全局属性，不是 HTML 标签；可添加到适用的已有元素上。

## 2 展开补充内容

### 2.1 &lt;details&gt; 与 &lt;summary&gt;

&lt;details&gt; 适合先显示问题、按需展开答案。它的第一个子元素是 &lt;summary&gt;，后面放补充内容，不需要自己编写点击脚本。

```html
<details>
  <summary>什么时候记录？</summary>
  <p>读完一个自然段后，先记下一个问题。</p>
</details>
<!-- 独立问题初始折叠；按 Tab 聚焦摘要，再按空格展开。 -->
```

配套文件：[scripts/12-native-interactive-elements/details.html](scripts/12-native-interactive-elements/details.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/details.html)

- 不写 open：初始折叠。
- 写 open：初始展开。
- open 是布尔属性；open="false" 仍表示展开。

保留摘要的原生键盘操作即可，不必再包一个按钮。&lt;details&gt; 表达展开关系，不是选项卡或菜单的通用替代品。

### 2.2 name：互斥的折叠组

同一元素树中，相同且非空的 name 使多个 &lt;details&gt; 属于同组；打开一项会关闭同组其他项。

reading-faq 是本例自定组名，不是内建关键字，也不是读者听到的控件名称。入口文字来自 &lt;summary&gt;。

- 同组初始最多给一项写 open。
- 同组 &lt;details&gt; 不相互嵌套。
- 需要同时比较多个答案时，省略 name，让各项独立展开。

```html
<details name="reading-faq" open>
  <summary id="length-summary">记录要写多长？</summary>
  <p>先写一个发现和一个仍想追问的问题。</p>
  <p><a href="#after-faq">读完这一项</a></p>
</details>
<details name="reading-faq">
  <summary id="topic-summary">没有想好主题怎么办？</summary>
  <p>挑出刚才最想再读一次的句子，从它开始。</p>
  <p><a href="#after-faq">读完这一项</a></p>
</details>
<!-- 检查：互斥组第一项初始展开；打开第二项后，第一项关闭。 -->
<!-- 用 Tab 将焦点移到 summary，按空格展开或折叠，再用 Tab 前进。 -->
<!-- 折叠项里的链接不应进入当前 Tab 顺序；展开后链接可以获得焦点。 -->
<!-- 浏览器不支持 name 时，可能同时展开两项；不能把这种表现当作互斥成功。 -->
```

配套文件：[scripts/12-native-interactive-elements/details.html](scripts/12-native-interactive-elements/details.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/details.html)

## 3 声明式对话框

### 3.1 打开和关闭模态对话框

模态（modal）对话框打开时，同一文档的其他内容暂不能交互。支持 Invoker Commands API 的浏览器可直接使用按钮属性：

- commandfor：目标 &lt;dialog&gt; 的 id，不加 #。
- command="show-modal"：模态打开。
- command="close"：关闭目标。

request-dialog 是本例对话框的 id。打开按钮位于框外，关闭按钮位于框内。

```html
<button id="open-request" type="button"
  commandfor="request-dialog" command="show-modal">允许 Esc</button>
<dialog id="request-dialog" closedby="closerequest"
  aria-labelledby="request-title">
  <h2 id="request-title">允许关闭请求</h2>
  <p>读完提示后可关闭。</p>
  <button id="close-request" type="button" autofocus
    commandfor="request-dialog" command="close">关闭提示</button>
</dialog>
```

配套文件：[scripts/12-native-interactive-elements/dialog-command.html](scripts/12-native-interactive-elements/dialog-command.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/dialog-command.html)

模态对话框进入顶层（top layer），即浏览器专门呈现浮层的显示层。autofocus 指定本例的初始焦点为关闭按钮；不要给 &lt;dialog&gt; 本身加 tabindex。

aria-labelledby 通过标题 id 为对话框提供可访问名称，本例 request-title 对应可见标题。模态与否取决于交互范围，不能靠居中、边框或焦点是否进入来判断。

### 3.2 closedby：允许哪些关闭方式

关闭请求（close request）通常由桌面 Esc 等平台操作发起。外部点击关闭（light dismiss）指点击或轻触对话框外部。

在浏览器支持 closedby，且脚本没有取消关闭请求时：

- closerequest：允许 Esc 等关闭请求，外部点击不关闭。
- any：同时允许关闭请求和外部点击。
- none：不启用上述自动关闭，仍可用明确的关闭按钮或 close()。

配套页有三个对话框供比较。下面这一个允许外部点击：

```html
<dialog id="any-dialog" closedby="any" aria-labelledby="any-title">
  <h2 id="any-title">允许外部点击</h2>
  <p>也可以点击对话框外部。</p>
  <button id="close-any" type="button" autofocus
    commandfor="any-dialog" command="close">关闭提示</button>
</dialog>
<!-- 每次只打开一个：初始焦点应在该框的关闭按钮；背景输入框不能编辑。 -->
<!-- closerequest：Esc 关闭，点外部不关闭；any：两种操作都可关闭。 -->
<!-- none：Esc 和外部点击均不关闭，但内部按钮仍能关闭。 -->
<!-- 每种关闭方式分别重开再测；键盘关闭后检查对应打开按钮的焦点轮廓。 -->
```

配套文件：[scripts/12-native-interactive-elements/dialog-command.html](scripts/12-native-interactive-elements/dialog-command.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/dialog-command.html)

省略 closedby 或值无效时，模态按 closerequest 处理，非模态按 none 处理。关闭入口应始终清楚可用。

用 Tab、Shift+Tab 检查焦点不能进入背景。用键盘打开并关闭本例后，检查焦点回到对应入口；实际应用若移除了入口，需要另选合理的焦点位置。

### 3.3 open：非模态显示与表单关闭

非模态（non-modal）允许继续操作外部页面。直接给 &lt;dialog&gt; 写 open 会初始非模态显示，不会停用背景，也没有一个刚刚触发它的按钮。

框内的 &lt;form method="dialog"&gt; 用于关闭对话框，不发送 HTTP 表单请求。提交按钮的 value 可成为对话框的 returnValue。

```html
<dialog open aria-labelledby="open-title">
  <h2 id="open-title">阅读提示</h2>
  <p>先保存一个问题，稍后再补充。</p>
  <form method="dialog">
    <button type="submit" value="understood">知道了</button>
  </form>
</dialog>
<!-- 预期：对话框初始可见，外部输入框仍能点击和编辑。 -->
<!-- 本例没有 closedby；非模态时缺省不响应 Esc，也不因外部点击关闭。 -->
<!-- “知道了”通过 method="dialog" 关闭，页面不应导航或提交数据。 -->
<!-- 静态 open 没有记录打开前的焦点，不把关闭后的焦点当作恢复触发按钮的示例。 -->
```

配套文件：[scripts/12-native-interactive-elements/dialog-open.html](scripts/12-native-interactive-elements/dialog-open.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/dialog-open.html)

关闭后刷新页面会重新读取 open。需要动态打开时，使用前面的命令按钮或下面的浏览器 API。

## 4 用浏览器 API 打开与关闭（补充）

需要由脚本决定何时打开时，&lt;dialog&gt; 提供以下浏览器 DOM API；它们不是 JavaScript 语言自身的内置函数。

- dialog.show()：非模态打开。
- dialog.showModal()：模态打开。
- dialog.close()：按对话框关闭流程关闭。

这里 dialog 是保存目标元素的变量。先关闭再切换模式，不能把已经非模态打开的框直接用 showModal() 升级为模态；也不要删除 open 或只改 CSS 来代替关闭。

配套页的两个按钮由脚本绑定不同动作：

```html
<button id="open-modeless" type="button">非模态打开</button>
<button id="open-modal" type="button">模态打开</button>
```

配套文件：[scripts/12-native-interactive-elements/dialog-api.html](scripts/12-native-interactive-elements/dialog-api.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/dialog-api.html)

querySelector 按选择器找到元素，addEventListener 注册点击处理。下面的 #note-dialog 表示 id 为 note-dialog；!dialog.open 用来避免重复打开。

```javascript
const dialog = document.querySelector("#note-dialog");
document.querySelector("#open-modeless").addEventListener("click", () => {
  if (!dialog.open) dialog.show();
});
```

配套文件：[scripts/12-native-interactive-elements/dialog-api.html](scripts/12-native-interactive-elements/dialog-api.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/dialog-api.html)

完整文件对另一个按钮调用 showModal()，对内部关闭按钮调用 close()，这些脚本已随页提供。

本页不设置 closedby。非模态允许编辑背景，Esc 默认不关闭；模态阻止背景交互，Esc 可关闭。用键盘打开时，两种模式都通过 autofocus 把初始焦点放到关闭按钮。

关闭时，模态会尝试恢复打开前的焦点；非模态需满足关闭时焦点仍在框内等条件，不能无条件把焦点从用户已选择的外部控件抢回来。

## 5 非模态弹出层

### 5.1 popover 与触发按钮

popover 是全局属性，使内容按需在顶层显示。它本身不产生模态效果，也不自动赋予菜单或对话框语义。本例用有标题的 &lt;section&gt; 表达补充说明。

- popovertarget：目标 id；目标同时需要 popover 属性。
- popovertargetaction="toggle"：切换显示与隐藏，也是省略此属性时的默认动作。
- popovertargetaction="show"：显示。
- popovertargetaction="hide"：隐藏。

下面按钮打开 help-panel。面板中的“什么是发现？”用于下一节的附属提示。

```html
<button id="open-help" type="button" popovertarget="help-panel">记录帮助</button>
<section id="help-panel" popover="auto" aria-labelledby="help-title">
  <h2 id="help-title">记录帮助</h2>
  <p>写下一个发现，再留下一个问题。</p>
  <button id="close-help" type="button"
    popovertarget="help-panel" popovertargetaction="hide">关闭帮助</button>
  <button id="open-hint" type="button" popovertarget="word-hint">什么是发现？</button>
</section>
```

配套文件：[scripts/12-native-interactive-elements/popover-auto.html](scripts/12-native-interactive-elements/popover-auto.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/popover-auto.html)

popover="auto" 允许外部点击及 Esc 等关闭请求。只写 popover 等同于 auto；省略整个属性则没有弹出行为。

通过 popovertarget 建立关系后，弹出内容参与触发按钮之后的键盘顺序。本例没有 autofocus，打开时焦点先留在触发按钮，Tab 再进入内部按钮。

### 5.2 独立 auto 与附属 hint

打开另一独立 auto 通常会关闭前一个；存在嵌套或内部触发关系的面板可以共存，不能说“全页永远只能一个”。

popover="hint" 适合附属提示。显示时不主动关闭 auto，会关闭非祖先的其他 hint，也允许外部点击及关闭请求。本例在帮助内部激活“什么是发现？”，显示这个提示：

```html
<section id="word-hint" popover="hint" aria-labelledby="hint-title">
  <h2 id="hint-title">发现</h2>
  <p>例如：重读一遍后，对同一句话有了不同理解。</p>
  <button type="button"
    popovertarget="word-hint" popovertargetaction="hide">关闭词语说明</button>
</section>
<!-- 不打开 hint 时，在 close-help 上按 Esc：帮助隐藏，焦点应回到 open-help。 -->
<!-- 再打开帮助，点击外部输入框：帮助消失，输入框仍能编辑，说明不是模态。 -->
<!-- 先开帮助再开例子：两个独立 auto 不应同时显示。 -->
<!-- 在帮助内打开“什么是发现？”：支持 hint 时帮助仍打开；可先关闭词语说明。 -->
<!-- 不把嵌套提示关闭后的焦点位置等同于前面的独立 auto 实验。 -->
```

配套文件：[scripts/12-native-interactive-elements/popover-auto.html](scripts/12-native-interactive-elements/popover-auto.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/popover-auto.html)

hint 不会仅凭该属性自动在悬停时显示，本例使用按钮触发。普通说明面板也不是具备菜单角色和方向键操作的完整菜单。

简单独立 auto 中，焦点仍在内部时按 Esc，可恢复到打开前聚焦的入口；用户已点击外部输入框时，不应把关闭理解为必须再抢回焦点。嵌套提示的关闭流程不能直接套用这一独立示例。

### 5.3 manual：明确关闭与焦点去向

popover="manual" 仍是非模态，但不会因外部点击、Esc 或另一个弹出层出现而自动关闭，多个独立 manual 可以同时显示。manual 控制关闭策略，不表示必须用脚本打开。

本例仍用声明式 show/hide 按钮：

```html
<button id="open-manual" type="button"
  popovertarget="manual-note" popovertargetaction="show">显示固定提示</button>
<section id="manual-note" popover="manual" aria-labelledby="manual-title">
  <h2 id="manual-title">固定提示</h2>
  <p>先写一个问题，再决定是否继续。</p>
  <button id="close-manual" type="button"
    popovertarget="manual-note" popovertargetaction="hide">关闭固定提示</button>
</section>
```

配套文件：[scripts/12-native-interactive-elements/popover-manual.html](scripts/12-native-interactive-elements/popover-manual.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/popover-manual.html)

manual 不提供与独立 auto 相同的焦点恢复流程。本例关闭按钮先调用 focus()，将焦点送回保留在外部的入口，再由 HTML 按钮动作隐藏面板。opener 保存打开按钮，focus() 是浏览器 DOM API。

```javascript
const opener = document.querySelector("#open-manual");
document.querySelector("#close-manual").addEventListener("click", () => {
  // 显式决定关闭后的去处；显示/隐藏仍由按钮的 HTML 属性完成。
  opener.focus();
});
```

配套文件：[scripts/12-native-interactive-elements/popover-manual.html](scripts/12-native-interactive-elements/popover-manual.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/popover-manual.html)

测试时分别尝试外部输入框、Esc 和内部关闭按钮，既检查面板是否隐藏，也检查键盘操作能否继续。

## 6 隐藏与停用

### 6.1 hidden：当前不呈现的内容

hidden 的普通隐藏状态使内容不呈现，也不作为普通内容向辅助技术提供。它是枚举属性；只写 hidden、写空值或 hidden="false" 都不能表示取消隐藏。

配套页把打开按钮留在隐藏区域外：

```html
<button id="reveal-extra" type="button">查看补充说明</button>
<section id="extra-note" hidden aria-labelledby="extra-title">
  <h3 id="extra-title">当前补充说明</h3>
  <p>本次只整理自己正在阅读的内容。</p>
  <button id="hide-extra" type="button">收起补充说明</button>
</section>
```

配套文件：[scripts/12-native-interactive-elements/hidden-inert.html](scripts/12-native-interactive-elements/hidden-inert.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/hidden-inert.html)

辅助脚本通过 hidden=false 揭示内容，通过 hidden=true 收起。这里 false 是 DOM 属性的布尔值，不是 HTML 属性里的字符串。收起前先把焦点移回外部入口：

```javascript
hide.addEventListener("click", () => {
  // 先将焦点移出将被隐藏的子树，再隐藏，保留可继续操作的入口。
  reveal.focus();
  extra.hidden = true;
});
```

配套文件：[scripts/12-native-interactive-elements/hidden-inert.html](scripts/12-native-interactive-elements/hidden-inert.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/hidden-inert.html)

extra 保存补充区域，reveal 保存外部打开按钮；完整绑定见配套文件。不要用 display:block 等样式覆盖隐藏效果。hidden 不会停止后代脚本执行，也不自动排除其中的表单控件提交。

### 6.2 inert：仍可见但暂不可交互

inert 是布尔属性，使区域及其后代暂不可点击、不可聚焦，并从可访问树中移除，默认仍可见。inert="false" 仍启用该属性。

编辑区的启用开关留在外部，否则使用者无法通过它恢复操作。

```html
<p><label><input id="enable-edit" type="checkbox">允许编辑</label></p>
<p id="edit-state" role="status">编辑区暂不可交互。</p>
<div id="edit-area" inert>
  <p><label for="draft-text">记录内容：</label>
    <input id="draft-text" value="留一个问题"></p>
  <p><a href="#archive-title">前往归档标题</a></p>
</div>
```

配套文件：[scripts/12-native-interactive-elements/hidden-inert.html](scripts/12-native-interactive-elements/hidden-inert.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/hidden-inert.html)

辅助脚本读取复选框的 checked，再设置 editArea.inert；外部 role="status" 提示可用状态，不需要移动焦点去读提示。用 Tab 与点击比较停用、启用两个状态，并在开发者工具的 Accessibility 中比较可访问树。

inert 不等同于表单 disabled，不是表单提交开关。模态对话框会让背景产生 inert 效果，不需要给每个背景元素插入 inert 属性。showModal() 打开的框可越过祖先的 inert，但框自身带 inert 仍会停用它。

### 6.3 hidden="until-found"：查找后显示（补充）

支持此取值时，页内查找或片段导航可以揭示内容，并移除 hidden。它不同于普通 hidden：隐藏时仍可能保留边框、内外边距所占的布局盒。

本例使用普通块级 &lt;section&gt;，不覆盖 display；若设成 none、contents 或 inline，会影响查找揭示。

```html
<p><a href="#archived-note">查找归档说明</a></p>
<section id="archived-note" hidden="until-found">
  <h3>归档说明</h3>
  <p>纸鹤归档：较早的记录按主题保存。</p>
</section>
<!-- 点击“查找归档说明”或用 Ctrl+F 查找“纸鹤归档”：支持 until-found 时内容显现。 -->
<!-- 归档被揭示后 hidden 被移除；刷新可恢复文件的初始状态。 -->
```

配套文件：[scripts/12-native-interactive-elements/hidden-inert.html](scripts/12-native-interactive-elements/hidden-inert.html) · [浏览器预览](http://127.0.0.1:8012/scripts/12-native-interactive-elements/hidden-inert.html)

浏览器不支持 until-found 时，内容可能按普通 hidden 保持隐藏；不能据此认定片段目标不存在。

## 本章小结

- &lt;details&gt; 组织可展开内容，&lt;summary&gt; 提供入口；相同 name 可建立互斥组。
- 模态对话框限制同一文档的背景交互；open 或 show() 的非模态显示不具备这一效果。
- popover 是属性，auto、hint、manual 的关闭与共存规则不同，不能只看面板外观。
- hidden 侧重不呈现，inert 侧重不可交互；关闭或停用时同时检查显示状态、焦点和可访问性。

## 练习

在 scripts/12-native-interactive-elements/ 内复制对应页面后修改。

（1）复制 details.html 为 practice-details.html，给互斥组增加第三个问题。检查：同组初始只有第一项展开；键盘打开第三项后其他项关闭。再去掉组内 name，确认两项可以同时展开。

（2）复制 dialog-command.html 为 practice-dialog.html，修改其中一个 closedby 值，先预测再分别测试关闭按钮、Esc 和外部点击。检查：记录各方式能否关闭、模态背景是否可编辑，以及键盘关闭后的焦点。

（3）复制 popover-auto.html 为 practice-popover.html，移除 hint 按钮与面板，把帮助改成 manual。检查：外部输入框仍可编辑，外部点击与 Esc 不关闭帮助；内部按钮可以关闭并把焦点送回入口。

（4）复制 hidden-inert.html 为 practice-hidden.html，在编辑区增加一个有标签的输入框。检查：停用时两个输入框都不能点击或 Tab 聚焦，启用后都能编辑；补充说明收起后焦点回到外部入口，并比较可访问树。

### 提示

第一题保留每项开头的 &lt;summary&gt;。第二题每次重新打开后只测一种关闭方式，不支持某属性时记录支持条件。第三题参考 manual 页的 focus() 调用。第四题将开关留在停用区域外。练习副本用后可删除。

## 参考与引用来源

- WHATWG HTML：[details](https://html.spec.whatwg.org/multipage/interactive-elements.html#the-details-element)与 [summary](https://html.spec.whatwg.org/multipage/interactive-elements.html#the-summary-element)的结构、open 和 name；[dialog](https://html.spec.whatwg.org/multipage/interactive-elements.html#the-dialog-element)的模态、命令按钮、closedby、dialog focusing steps 与 close the dialog；[popover](https://html.spec.whatwg.org/multipage/popover.html#the-popover-attribute)的显示、隐藏、焦点恢复及 auto/hint/manual 关系；[hidden](https://html.spec.whatwg.org/multipage/interaction.html#the-hidden-attribute)的状态、布局和查找揭示；[Inert subtrees](https://html.spec.whatwg.org/multipage/interaction.html#inert-subtrees)的不可交互、模态背景与祖先例外。
- MDN：[dialog](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/dialog#attributes)、[button 的 command](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/button#command)、[showModal()](https://developer.mozilla.org/en-US/docs/Web/API/HTMLDialogElement/showModal)的打开条件与 Browser compatibility；[Using the Popover API](https://developer.mozilla.org/en-US/docs/Web/API/Popover_API/Using#popover_accessibility_features)、[popover 值](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Global_attributes/popover#value)的声明式操作、焦点顺序和 hint；[inert](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Global_attributes/inert#accessibility_concerns)的可访问性与 [focus()](https://developer.mozilla.org/en-US/docs/Web/API/HTMLElement/focus)的焦点管理；[querySelector](https://developer.mozilla.org/en-US/docs/Web/API/Document/querySelector)、[addEventListener](https://developer.mozilla.org/en-US/docs/Web/API/EventTarget/addEventListener)、[HTMLElement.hidden](https://developer.mozilla.org/en-US/docs/Web/API/HTMLElement/hidden)支持必要的元素查找、事件绑定和隐藏切换。
- W3C WAI：[Dialog (Modal) Pattern](https://www.w3.org/WAI/ARIA/apg/patterns/dialog-modal/)的键盘交互、初始焦点、关闭后焦点去向及标题命名；具体原生关闭条件以 HTML 规则为准。
- Chrome for Developers：[Accessibility features reference](https://developer.chrome.com/docs/devtools/accessibility/reference/#tab)：可访问树与控件属性检查。
- Python 3.12：[http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface)：服务目录、端口及绑定地址。